In [2]:
Section 1: Load & Profile the Raw Data
Loaded the files and look at their structure to identify missing values, duplicates, and formatting issues.

In [81]:
import pandas as pd

In [82]:
df_sales = pd.read_csv("sales_messy.csv") # Load the raw datasets

In [83]:
df_cust = pd.read_csv("customers.csv")

In [84]:
print("Sales Data Shape") # Profile raw data structures
print(df_sales.shape)

Sales Data Shape
(208, 9)


In [85]:
print("Sales Information")
df_sales.info()

Sales Information
<class 'pandas.DataFrame'>
RangeIndex: 208 entries, 0 to 207
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   order_id     208 non-null    int64  
 1   order_date   208 non-null    str    
 2   customer_id  200 non-null    float64
 3   country      208 non-null    str    
 4   category     208 non-null    str    
 5   product      208 non-null    str    
 6   quantity     208 non-null    int64  
 7   unit_price   195 non-null    float64
 8   discount     188 non-null    float64
dtypes: float64(3), int64(2), str(4)
memory usage: 21.7 KB


In [86]:
print("Missing Value Counts")
print(df_sales.isnull().sum())

Missing Value Counts
order_id        0
order_date      0
customer_id     8
country         0
category        0
product         0
quantity        0
unit_price     13
discount       20
dtype: int64


In [87]:
print("Total Duplicate Rows")
print(df_sales.duplicated().sum())

Total Duplicate Rows
8


In [88]:
print("Unique Countries Before Cleaning")
print(df_sales["country"].unique())

Unique Countries Before Cleaning
<ArrowStringArray>
[     'GERMANY',      'Germany',       'France',      ' France',
   'Kazakhstan',           'UK', ' kazakhstan ',          'uk ',
       'Poland',          'usa',          'USA',       'Russia']
Length: 12, dtype: str


In [ ]:
Section 2: Clean the Data
This section fixes dataset by deleting duplicates, trimming text inconsistencies, and filling missing blocks logically.

In [89]:
df_sales = df_sales.drop_duplicates() # 1.Remove duplicate entries

In [90]:
df_sales = df_sales.dropna(subset=["customer_id"]) # 2.Drop rows that have no customer_id

In [91]:
df_sales["country"] = df_sales["country"].astype(str).str.strip().str.title() # 3.Clean and standardise country names
df_sales["country"] = df_sales["country"].replace({"Uk": "UK", "Usa": "USA"})

In [92]:
df_sales["discount"] = df_sales["discount"].fillna(0.0) # 4.Fill missing discount values with 0

In [93]:
median_price = df_sales["unit_price"].median() # 5.Fill missing unit prices using the dataset median price
df_sales["unit_price"] = df_sales["unit_price"].fillna(median_price)

In [94]:
df_sales["order_date"] = pd.to_datetime(df_sales["order_date"].astype(str).str.strip()) # 6.Parse order_date to proper datetime format

In [95]:
# Verify zero missing values in our targeted columns
print("Missing Values After Cleaning")
print(df_sales[["customer_id", "country", "discount", "unit_price", "order_date"]].isnull().sum())

Missing Values After Cleaning
customer_id    0
country        0
discount       0
unit_price     0
order_date     0
dtype: int64


In [52]:
Section 3: Enrich Data with Computed Columns
Added derived columns to compute total revenue and extract the month number from the transaction date.

SyntaxError: invalid syntax (2698892228.py, line 1)

In [96]:
df_sales["revenue"] = df_sales["quantity"] * df_sales["unit_price"] * (1 - df_sales["discount"]) # Calculate financial metrics and extract months
df_sales["month"] = df_sales["order_date"].dt.month

In [97]:
df_sales[["order_date", "month", "quantity", "unit_price", "discount", "revenue"]].head(5) # Preview first 5 rows to check calculations

,order_date,month,quantity,unit_price,discount,revenue
1,2025-07-24,7,3,59.99,0.10,161.973
2,2025-02-16,2,1,799.00,0.05,759.050
3,2025-12-15,12,4,899.00,0.20,2876.800
4,2025-08-28,8,5,549.00,0.10,2470.500
5,2025-02-28,2,6,59.99,0.15,305.949


In [ ]:
Section 4: Merge Datasets
Used a left-join to bring customer demographic segments into sales table without changing the original row count.

In [98]:
rows_before = len(df_sales) # Save the row count before merging

In [99]:
df_merged = pd.merge(df_sales, df_cust, on="customer_id", how="left") # Merge datasets using a left join on customer_id

In [100]:
# Check if row count remained identical
print("Merge Integrity Verification")
print(f"Rows Before Merge: {rows_before}")
print(f"Rows After Merge:  {len(df_merged)}")
print(f"Is row count unchanged? {rows_before == len(df_merged)}")

Merge Integrity Verification
Rows Before Merge: 193
Rows After Merge:  193
Is row count unchanged? True


In [ ]:
Section 5: Aggregate Data
Summarized and group the data into sorted tables to answer strategic business questions.

In [105]:
# Isolate global net revenue for percentage calculations
grand_total = df_merged["revenue"].sum()
print(f"Grand Total Revenue: ${grand_total:,.2f}")

Grand Total Revenue: $293,209.90


In [102]:
print("Total Revenue per Category (Sorted)")
cat_summary = df_merged.groupby("category")["revenue"].sum().reset_index()
cat_summary = cat_summary.sort_values(by="revenue", ascending=False).reset_index(drop=True)
cat_summary["share_%"] = (cat_summary["revenue"] / grand_total) * 100
print(cat_summary)

Total Revenue per Category (Sorted)
      category      revenue    share_%
0      Laptops  161187.4000  54.973383
1       Phones   63403.4000  21.623895
2     Monitors   58295.5500  19.881849
3  Accessories   10323.5465   3.520872


In [103]:
print("Total Revenue per Month (Sorted)")
month_summary = df_merged.groupby("month")["revenue"].sum().reset_index()
month_summary = month_summary.sort_values(by="revenue", ascending=False).reset_index(drop=True)
print(month_summary)

Total Revenue per Month (Sorted)
    month     revenue
0       7  42529.4260
1      10  33697.7500
2       8  30827.4315
3       4  26456.2405
4       6  24754.8375
5       5  23633.5065
6      12  22739.7510
7       3  19836.5880
8       2  19631.0780
9      11  19117.3905
10      1  15348.3950
11      9  14637.5020


In [104]:
print("Total Revenue per Customer Segment (Sorted)")
segment_summary = df_merged.groupby("segment")["revenue"].sum().reset_index()
segment_summary = segment_summary.sort_values(by="revenue", ascending=False).reset_index(drop=True)
print(segment_summary)

Total Revenue per Customer Segment (Sorted)
     segment      revenue
0   Consumer  174850.8365
1  Education   77782.0320
2   Business   40577.0280


In [ ]:
Section 6: Conclusions
Core business insights derived directly from sorted summary data outputs.

In [ ]:
1) Dominant Category & Share: The Laptops category is the largest driver of business value, 
generating a total revenue of $161,187.40, which represents a 54.97% majority share of our overall sales volume ($293,209.90).
2) Best Trading Month: July (Month 7) is the highest-earning period in our timeline, hitting a seasonal high revenue milestone of $42,529.43.
3) Leading Market Segment: The Consumer demographic segment serves as our main client engine, bringing in $174,850.84 of total revenue.
4) Surprising Observation: Despite heavy daily usage in modern work-from-home and office jobs, 
the Accessories vertical was our worst-performing product group, ranking dead last with a minimal total revenue of only $10,323.55.